<a href="https://colab.research.google.com/github/Ashwathi1901/Bug-Bounty-Agentic-AI/blob/main/Exploit_Probability_Calculation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

from google.colab import files
uploaded = files.upload()

In [ ]:
df = pd.read_csv("processed_vulnerabilities.csv")

In [ ]:
!pip install transformers torch
from transformers import pipeline

classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

In [ ]:
severity_map = {
    "LOW": 0.2,
    "MEDIUM": 0.5,
    "HIGH": 0.8,
    "CRITICAL": 1.0
}
class_risk = {
    "SQL Injection": 1.0,
    "Remote Code Execution": 1.0,
    "Command Injection": 0.95,
    "Cross Site Scripting": 0.7,
    "Directory Traversal": 0.6,
    "Authentication Bypass": 0.9,
    "Buffer Overflow": 0.85,
    "Privilege Escalation": 0.9,
    "Improper Access Control": 0.8,
    "Information Disclosure": 0.5,
    "Denial of Service": 0.6,
    "Race Condition": 0.7,
    "Use After Free": 0.85,
    "Integer Overflow": 0.8,
    "Security Misconfiguration": 0.5
}

cwe_risk = {
    89: 1.0,   # SQL Injection
    79: 0.7,   # Cross Site Scripting
    120: 0.85, # Buffer Overflow
    269: 0.9,  # Privilege Escalation
    22: 0.6,   # Directory Traversal
    287: 0.9,  # Authentication Bypass
    284: 0.8,  # Improper Access Control
    200: 0.5,  # Information Disclosure
    400: 0.6,  # Denial of Service
    362: 0.7,  # Race Condition
    94: 0.95,  # Code Injection
    416: 0.85, # Use After Free
    190: 0.8,  # Integer Overflow
    16: 0.5,   # Security Misconfiguration
    77: 0.95   # Command Injection
}

In [ ]:
print("Calculating Exploit Vulnerability in real time of your submitted report.....")

def calculate_exploit(row):
    sev_score = severity_map.get(row['SEVERITY'].upper(), 0.5)
    class_score = class_risk.get(row['Class'], 0.5)

    cwe_num = int(row['CWE-ID']) if str(row['CWE-ID']).isdigit() else 0
    cwe_score = cwe_risk.get(cwe_num, 0.5)

    prob = (0.4 * sev_score + 0.4 * class_score + 0.2 * cwe_score) * row['Confidence']
    prob = round(prob, 2)

    if prob < 0.3:
        risk = "Low"
    elif prob < 0.6:
        risk = "Medium"
    elif prob < 0.8:
        risk = "High"
    else:
        risk = "Critical"

    return f"{prob} ({risk})"

In [ ]:
df["exploit_vulnerability"] = df.apply(calculate_exploit, axis=1)